[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mbanuelos/grad_math_modeling/blob/main/Lectures/Module0_TimeSeries/11_ExogenousVariables.ipynb)

# Exogenous Variables and the Bike-Rental Series

**Module 0 · Lesson 11 of 13 · Student edition**  
**Estimated class time:** 75–90 minutes  
**Source sequence:** Original Day 4  

**Prerequisite:** Lessons 1–3  

## Learning objectives

By the end of this lesson, you should be able to:

- Identify plausible external predictors for a forecasting problem.
- Explore a new target and its candidate exogenous variables.
- Repeat the stationarity workflow on daily data.

## Before We Begin

> **Prompt:** Think of something you predict regularly that depends on information
> *outside* its own past — not just "what happened last time," but some other signal
> entirely.

> Write 3–5 sentences describing this prediction and the *external* signal you use.
> Then sketch a simple diagram showing the external signal feeding into your prediction alongside the thing's own past behavior.


---
### Quick Recap from Days 1–3

- **Day 1:** Diagnosed time series — stationarity, transformations, ACF/PACF.
- **Day 2:** Built AR models and the **lag-embedded feature matrix** — re-framing
  forecasting as supervised learning.
- **Day 3:** Implemented six model families (naive, AR, Random Forest, Gradient
  Boosting, MLP, RNN, LSTM) with a **time-aware train/test split** and compared them
  using **RMSE** and **MAE**.

So far, every model has predicted a series using **only its own past values**. But real
forecasting problems often have **external information** available — weather, holidays,
economic indicators — that can improve predictions beyond what the series' own history
tells you.

Across Lessons 11–13 you will:
1. Build a lag matrix that incorporates **external (exogenous) variables**
2. Compare a univariate model to an exogenous-aware model on a real dataset
3. Revisit the MLP, now with exogenous features included
4. Build a complete model comparison table across the whole week's model families
5. Write a capstone reflection connecting every model's notion of "memory"


---
## Part 0 — Imports & Setup

Run the cell below. You do **not** need to modify it.


In [ ]:
# Imports used in this lesson
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
%matplotlib inline

print('All imports successful!')


---
## Part 1 — A New Dataset: Daily Bike Rentals

Today we switch datasets. The **Capital Bikeshare** daily rental counts (Washington,
D.C., 2011–2012) come with genuine external variables recorded alongside the target:
temperature, humidity, windspeed, and whether the day was a working day or holiday.
This is a much more natural setting for exogenous variables than Air Passengers, which
only ever gave us the series itself.

### 1.1 Load the data

This cell is complete.


In [3]:
url = 'https://raw.githubusercontent.com/christophM/interpretable-ml-book/master/data/bike-sharing-daily.csv'
df = pd.read_csv(url, parse_dates=['dteday'])

# Keep only the columns we need today, with clearer names
df = df[['dteday', 'cnt', 'temp', 'hum', 'windspeed', 'workingday', 'holiday']].copy()
df.columns = ['Date', 'Rentals', 'Temp', 'Humidity', 'Windspeed', 'WorkingDay', 'Holiday']

print('Shape:', df.shape)
df.head()


Shape: (731, 7)


,Date,Rentals,Temp,Humidity,Windspeed,WorkingDay,Holiday
0,2011-01-01,985,0.344167,0.805833,0.160446,0,0
1,2011-01-02,801,0.363478,0.696087,0.248539,0,0
2,2011-01-03,1349,0.196364,0.437273,0.248309,1,0
3,2011-01-04,1562,0.200000,0.590435,0.160296,1,0
4,2011-01-05,1600,0.226957,0.436957,0.186900,1,0


> 💡 **Note on the variables:** `Temp`, `Humidity`, and `Windspeed` are already
> normalized to roughly $[0, 1]$ by the dataset's creators. `WorkingDay` and `Holiday`
> are binary flags (1 = yes, 0 = no). `Rentals` is the total daily bike rental count —
> our target.


### 1.2 Visualize the target and one candidate exogenous variable

**Your turn!** Plot `Rentals` over time, and below it, plot `Temp` over the same
time range. Use two stacked subplots sharing the x-axis.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# FILL IN: plot df['Rentals'] against df['Date'] on the first axis
axes[0].plot(df['???'], df['???'], color='steelblue')
axes[0].set_title('Daily Bike Rentals')

# FILL IN: plot df['Temp'] against df['Date'] on the second axis
axes[1].plot(df['???'], df['???'], color='darkorange')
axes[1].set_title('Normalized Temperature')

plt.tight_layout()
plt.show()


### ✏️ Written Response 1.2

1. Do `Rentals` and `Temp` appear to rise and fall together, or do they move
   independently? Describe what you see.
2. Does this match your intuition about bike rentals and weather? Why might
   temperature be a useful **external** predictor for rental counts?
3. `Rentals` also shows a clear trend across the two years. What might explain that,
   beyond weather alone? *(Hint: this is a bike-share program in its early years.)*

> **YOUR ANSWER:**


### 1.3 Quantify the relationship

This cell is complete, run it to see the correlation between `Rentals` and each
candidate exogenous variable.


In [4]:
exog_candidates = ['Temp', 'Humidity', 'Windspeed', 'WorkingDay', 'Holiday']

correlations = df[['Rentals'] + exog_candidates].corr()['Rentals'].drop('Rentals')
print('Correlation of each candidate variable with Rentals:')
print(correlations.sort_values(ascending=False))


Correlation of each candidate variable with Rentals:
Temp          0.627494
WorkingDay    0.061156
Holiday      -0.068348
Humidity     -0.100659
Windspeed    -0.234545
Name: Rentals, dtype: float64


### ✏️ Written Response 1.3

1. Which variable has the strongest correlation with `Rentals`? Is the sign
   (positive/negative) what you expected?
2. `Humidity` and `Windspeed` typically show weak negative correlations. Propose a
   real-world explanation for why higher humidity or wind might *reduce* rentals.
3. Based on this table alone, which variable would you choose as your primary
   exogenous predictor today?

> **YOUR ANSWER:**


---
## Part 2 — Quick Stationarity Check

Before building any model, run the Day 1 diagnostic habit on the new target series.

**Your turn!** Reuse `run_adf` (rewritten below for convenience) to check whether
`Rentals` is stationary.


In [ ]:
from statsmodels.tsa.stattools import adfuller

def run_adf(series, label='Series'):
    result = adfuller(series.dropna())
    print(f'ADF Test: {label}')
    print(f'  p-value: {result[1]:.4f}')
    print('  --> STATIONARY' if result[1] < 0.05 else '  --> NON-STATIONARY')
    print()

# FILL IN: run the ADF test on df['Rentals']
#run_adf(df[???], label='Daily Rentals')


> 💡 **Heads up:** Unlike Air Passengers, the bike rentals series does **not** need
> a log transform — there's no strong multiplicative seasonality here. But it may still
> need differencing if the ADF test says it's non-stationary. If so, apply a first
> difference to `Rentals` and re-test before continuing — exactly the Day 1 workflow,
> just applied to a new series.


In [10]:
# FILL IN: if needed, apply a first difference and re-test
# (If the original series is already stationary, you can skip this and use it directly —
#  just be consistent about which version you use for the rest of the notebook.)
df['Rentals_Diff'] = df['Rentals'].diff()

run_adf(df['Rentals_Diff'], label='Differenced Rentals')


ADF Test: Differenced Rentals
  p-value: 0.0000
  --> STATIONARY

